# ExomSystem — Pruebas DAO
Este notebook prueba todas las operaciones CRUD del proyecto CivicTech.

> **Requisito:** tener el `docker-compose up` corriendo antes de ejecutar.

## 0. Imports

In [6]:
import sys
import os

# Asegura que Python encuentra dao.py y db_models/ desde /app (raíz del proyecto)
sys.path.insert(0, '/app')

from datetime import datetime
from dao import ConexionDB, UsuarioDAO, ReporteDAO, EvidenciaDAO
from db_models import Usuario, Reporte, Evidencia

print('Imports OK ✅')
print('sys.path[0]:', sys.path[0])

Imports OK ✅
sys.path[0]: /app


## 1. Verificar conexión a la base de datos

In [7]:
# Las variables de entorno las inyecta docker-compose automáticamente
print('DB_HOST :', os.getenv('DB_HOST'))
print('DB_NAME :', os.getenv('DB_NAME'))
print('DB_USER :', os.getenv('DB_USER'))

with ConexionDB() as conn:
    print('\n✅ Conexión exitosa a PostgreSQL + PostGIS')

DB_HOST : db
DB_NAME : civictech_db
DB_USER : postgres

✅ Conexión exitosa a PostgreSQL + PostGIS


## 2. UsuarioDAO — CRUD

In [9]:
# ── Crear usuario ──────────────────────────────────────────
with ConexionDB() as conn:
    dao = UsuarioDAO(conn)
    usuario = dao.crear(Usuario(
        nombre_completo='Ana Pérez',
        dni='30123456',
        email='ana@civictech.com',
        contrasena='password123'
    ))
    print('Creado :', usuario)

UndefinedColumn: column "dni" of relation "usuario" does not exist
LINE 2: ...          INSERT INTO "usuario" (nombre_apellido, dni, email...
                                                             ^


In [10]:
# ── Obtener por ID ─────────────────────────────────────────
with ConexionDB() as conn:
    dao = UsuarioDAO(conn)
    u = dao.obtener_por_id(usuario.id_usuario)
    print('Por ID  :', u)

NameError: name 'usuario' is not defined

In [8]:
# ── Obtener por email ──────────────────────────────────────
with ConexionDB() as conn:
    dao = UsuarioDAO(conn)
    u = dao.obtener_por_email('ana@civictech.com')
    print('Por email:', u)

UndefinedTable: relation "usuarios" does not exist
LINE 3:             FROM usuarios
                         ^


In [9]:
# ── Actualizar reputación ──────────────────────────────────
with ConexionDB() as conn:
    dao = UsuarioDAO(conn)
    ok = dao.actualizar_reputacion(usuario.id_usuario, 85)
    print('Reputación actualizada:', ok)
    print('Nuevo score:', dao.obtener_por_id(usuario.id_usuario).reputacion_score)

NameError: name 'usuario' is not defined

In [10]:
# ── Listar todos ───────────────────────────────────────────
with ConexionDB() as conn:
    dao = UsuarioDAO(conn)
    todos = dao.obtener_todos()
    print(f'Total usuarios: {len(todos)}')
    for u in todos:
        print(' -', u)

UndefinedTable: relation "usuarios" does not exist
LINE 3:             FROM usuarios
                         ^


## 3. ReporteDAO — CRUD + PostGIS

In [6]:
# ── Crear reporte (coordenadas de Chilecito, La Rioja) ─────
# Tupla: (longitud, latitud)  →  PostGIS usa X=lon, Y=lat
CHILECITO = (-66.8541, -29.4967)

with ConexionDB() as conn:
    dao = ReporteDAO(conn)
    reporte = dao.crear(Reporte(
        id_usuario=usuario.id_usuario,
        id_tipo=1,                          # 'Estacionamiento en doble fila'
        patente_vehiculo='AB123CD',
        fecha_hora_dispositivo=datetime.now(),
        fecha_hora_servidor=datetime.now(),
        coordenadas_gps=CHILECITO,
    ))
    print('Creado:', reporte)

NameError: name 'usuario' is not defined

In [ ]:
# ── Obtener por ID ─────────────────────────────────────────
with ConexionDB() as conn:
    dao = ReporteDAO(conn)
    r = dao.obtener_por_id(reporte.id_reporte)
    print('Por ID :', r)
    print('GPS    :', r.coordenadas_gps)

In [ ]:
# ── Obtener por patente ────────────────────────────────────
with ConexionDB() as conn:
    dao = ReporteDAO(conn)
    lista = dao.obtener_por_patente('AB123CD')
    print(f'Reportes para AB123CD: {len(lista)}')
    for r in lista:
        print(' -', r)

In [ ]:
# ── Obtener por estado ─────────────────────────────────────
with ConexionDB() as conn:
    dao = ReporteDAO(conn)
    pendientes = dao.obtener_por_estado('Pendiente')
    print(f'Reportes pendientes: {len(pendientes)}')

In [ ]:
# ── Consulta espacial PostGIS: radio de 1 km ───────────────
with ConexionDB() as conn:
    dao = ReporteDAO(conn)
    cercanos = dao.obtener_en_radio(
        lon=-66.8541, lat=-29.4967, metros=1000
    )
    print(f'Reportes en radio 1 km: {len(cercanos)}')
    for r in cercanos:
        print(f'  id={r.id_reporte}  GPS={r.coordenadas_gps}  estado={r.estado_resolucion}')

In [ ]:
# ── Actualizar estado ──────────────────────────────────────
# Estados válidos: 'Pendiente' | 'En revisión' | 'Aprobado' | 'Rechazado'
with ConexionDB() as conn:
    dao = ReporteDAO(conn)
    ok = dao.actualizar_estado(reporte.id_reporte, 'En revisión')
    print('Estado actualizado:', ok)
    r = dao.obtener_por_id(reporte.id_reporte)
    print('Nuevo estado:', r.estado_resolucion)

## 4. EvidenciaDAO — CRUD

In [ ]:
import hashlib

# Simular hash SHA-256 de una foto
hash_foto = hashlib.sha256(b'foto_de_prueba_civictech').hexdigest()
print('Hash SHA-256:', hash_foto)

with ConexionDB() as conn:
    dao = EvidenciaDAO(conn)
    evidencia = dao.crear(Evidencia(
        id_reporte=reporte.id_reporte,
        url_archivo_s3='https://mi-bucket.s3.amazonaws.com/fotos/AB123CD_001.jpg',
        hash_seguridad_sha256=hash_foto,
    ))
    print('Creada:', evidencia)

In [ ]:
# ── Obtener por reporte ────────────────────────────────────
with ConexionDB() as conn:
    dao = EvidenciaDAO(conn)
    lista = dao.obtener_por_reporte(reporte.id_reporte)
    print(f'Evidencias del reporte {reporte.id_reporte}: {len(lista)}')
    for e in lista:
        print(' -', e)

In [ ]:
# ── Buscar por hash (validación de integridad jurídica) ────
with ConexionDB() as conn:
    dao = EvidenciaDAO(conn)
    e = dao.obtener_por_hash(hash_foto)
    print('Evidencia encontrada por hash:', e)

## 5. Limpieza (opcional)
Ejecutar solo si querés borrar los datos de prueba.

In [ ]:
# Las evidencias y reportes se eliminan en cascada al borrar el usuario
with ConexionDB() as conn:
    dao = UsuarioDAO(conn)
    ok = dao.eliminar(usuario.id_usuario)
    print('Usuario eliminado (cascade):', ok)